# 03 — Multi-tool agent workflow
Run one compound request and inspect the plan and structured trace.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == '02_notebooks' else Path.cwd().resolve()
SRC = ROOT / '03_src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))

import os, json
from dataclasses import asdict
os.environ['TRAVEL_DATA_MODE']='local'
from travel_agent.agent import PlanThenExecuteTravelAgent

In [2]:
query='I am going to Vienna for 3 days. Check weather, find hotels under 150 EUR, show museums and landmarks, public transport costs, and convert 500 EUR to HUF.'
run=PlanThenExecuteTravelAgent().run(query)
print(run.answer)

Weather (01_data/raw/weather_fallback.csv): 2026-09-12: 13.8–22.3°C, rain 0 mm; 2026-09-13: 16.3–24.8°C, rain 5.4 mm; 2026-09-14: 16.9–25.4°C, rain 1.8 mm
Currency: 500 EUR ≈ 197500 HUF (01_data/raw/fx_rates_fallback.csv).
Hotels: 15 matches. Top options: Market Vienna Rooms 60 (€150/night, 5.0/5), Urban Vienna Inn 39 (€142/night, 4.7/5), Urban Vienna Suites 27 (€145/night, 4.7/5).
Attractions: Vienna Museum 01 (museum, €0), Vienna Landmark 02 (landmark, €0), Vienna Museum 11 (museum, €0), Vienna Landmark 12 (landmark, €5).
Transport: day pass €9.00, 3-day pass €22.95; estimated 3-day pass cost €22.95.


In [3]:
run.metadata['plan']

[{'step': 1,
  'tool': 'get_weather',
  'reason': 'Weather requested.',
  'arguments': {'city': 'Vienna', 'days': 3, 'unit': 'celsius'},
  'depends_on': []},
 {'step': 2,
  'tool': 'convert_currency',
  'reason': 'Currency conversion requested.',
  'arguments': {'amount': 500.0, 'from_currency': 'EUR', 'to_currency': 'HUF'},
  'depends_on': []},
 {'step': 3,
  'tool': 'search_hotels',
  'reason': 'Accommodation search requested.',
  'arguments': {'city': 'Vienna',
   'nights': 3,
   'max_price_per_night_eur': 150.0,
   'min_rating': 0.0,
   'top_k': 5},
  'depends_on': []},
 {'step': 4,
  'tool': 'search_attractions',
  'reason': 'Attraction research requested.',
  'arguments': {'city': 'Vienna',
   'categories': ['museum', 'landmark'],
   'max_ticket_eur': None,
   'top_k': 6},
  'depends_on': []},
 {'step': 5,
  'tool': 'get_transport_options',
  'reason': 'Local transport requested.',
  'arguments': {'city': 'Vienna', 'days': 3},
  'depends_on': []}]

In [4]:
[asdict(x) for x in run.trace]

[{'step': 1,
  'name': 'get_weather',
  'arguments': {'city': 'Vienna', 'days': 3, 'unit': 'celsius'},
  'output': {'city': 'Vienna',
   'forecast': [{'date': '2026-09-12',
     'temp_min_c': 13.8,
     'temp_max_c': 22.3,
     'precipitation_mm': 0.0,
     'condition': 'clear',
     'wind_kph': 8.0},
    {'date': '2026-09-13',
     'temp_min_c': 16.3,
     'temp_max_c': 24.8,
     'precipitation_mm': 5.4,
     'condition': 'partly_cloudy',
     'wind_kph': 15.0},
    {'date': '2026-09-14',
     'temp_min_c': 16.9,
     'temp_max_c': 25.4,
     'precipitation_mm': 1.8,
     'condition': 'cloudy',
     'wind_kph': 22.0}],
   'source': '01_data/raw/weather_fallback.csv',
   'fallback': True},
  'latency_ms': 1.8230190000849689,
  'success': True,
  'error': None},
 {'step': 2,
  'name': 'convert_currency',
  'arguments': {'amount': 500.0, 'from_currency': 'EUR', 'to_currency': 'HUF'},
  'output': {'amount': 500.0,
   'from_currency': 'EUR',
   'to_currency': 'HUF',
   'converted_amount':

For `openai_direct`, the same registry is exposed as function schemas through the OpenAI Responses API. The difference is who creates the plan, not how the tools execute.